In [1]:
import time
import pickle

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

from tensorflow.keras import Sequential, Input
from tensorflow.keras.models import Sequential

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)

from tensorflow.keras.layers import (
    Embedding,
    Dense,
    SimpleRNN,
    LSTM,
    GRU,
    Bidirectional,
    GlobalAveragePooling1D,
    Dropout,
    BatchNormalization
)

from tensorflow.keras.preprocessing.sequence import pad_sequences

In [2]:
data = pd.read_csv("imdb_cleaned.csv")
print(data.shape)

(49582, 4)


In [3]:
X = data["clean_review"]
y = data["label"]

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_test,
    y_test,
    test_size=0.50,
    random_state=42,
    stratify=y_test
)

print(len(X_train))
print(len(X_val))
print(len(X_test))

39665
4958
4959


In [5]:
import pickle

with open("tokenizer.pkl", "rb") as file:
    tokenizer = pickle.load(file)

print(len(tokenizer.word_index))

90662


In [6]:
MAX_SEQUENCE_LENGTH = 500

X_train_sequences = tokenizer.texts_to_sequences(X_train)
X_val_sequences = tokenizer.texts_to_sequences(X_val)
X_test_sequences = tokenizer.texts_to_sequences(X_test)

X_train_integer = pad_sequences(
    X_train_sequences,
    maxlen=MAX_SEQUENCE_LENGTH,
    padding="post",
    truncating="post"
)

X_val_integer = pad_sequences(
    X_val_sequences,
    maxlen=MAX_SEQUENCE_LENGTH,
    padding="post",
    truncating="post"
)

X_test_integer = pad_sequences(
    X_test_sequences,
    maxlen=MAX_SEQUENCE_LENGTH,
    padding="post",
    truncating="post"
)

y_train = np.asarray(y_train)
y_val = np.asarray(y_val)
y_test = np.asarray(y_test)

print(X_train_integer.shape)
print(X_val_integer.shape)
print(X_test_integer.shape)

(39665, 500)
(4958, 500)
(4959, 500)


In [7]:
def evaluate_model(model, X_test, y_test, model_name, batch_size=64):

    # Generate probabilities
    probabilities = model.predict(
        X_test,
        batch_size=batch_size,
        verbose=0
    ).ravel()

    # Convert probabilities to class predictions
    predictions = (probabilities >= 0.5).astype(int)

    # metrics
    accuracy = accuracy_score(y_test, predictions)
    precision = precision_score(y_test, predictions, zero_division=0)
    recall = recall_score(y_test,  predictions, zero_division=0)
    f1 = f1_score(y_test, predictions, zero_division=0)
    roc_auc = roc_auc_score(y_test, probabilities)

    # Confusion Matrix
    cm = confusion_matrix(y_test, predictions)

    # Classification Report
    report = classification_report(
        y_test,
        predictions,
        target_names=["Negative", "Positive"],
        digits=4
    )

    # Print results
    print("=" * 60)
    print(f"{model_name} RESULTS")
    print("=" * 60)

    print(f"Accuracy : {accuracy:.5f}")
    print(f"Precision: {precision:.5f}")
    print(f"Recall   : {recall:.5f}")
    print(f"F1 Score : {f1:.5f}")
    print(f"ROC-AUC  : {roc_auc:.5f}")

    print("\nClassification Report")
    print("-" * 60)
    print(report)

    print("Confusion Matrix")
    print(cm)

    # Return everything for later comparison
    results = {
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "ROC-AUC": roc_auc
    }

    return results, probabilities, predictions, cm

In [8]:
rnn_results = []
rnn_results.append({
    "Model": "RNN Baseline",
    "Accuracy": 0.50534,
    "Precision": 0.59000,
    "Recall": 0.04741,
    "F1 Score": 0.08776,
    "ROC-AUC": 0.51143
})

rnn_df = pd.DataFrame(rnn_results)
display(rnn_df)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,RNN Baseline,0.50534,0.59,0.04741,0.08776,0.51143


**RNN EarlyStopping**

In [15]:
VOCAB_SIZE = 30000
EMBEDDING_DIM = 128
MAX_SEQUENCE_LENGTH = 500

In [16]:
rnn_early_model = Sequential([
    Input(
        shape=(MAX_SEQUENCE_LENGTH,),
        dtype="int32"
    ),

    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),
    SimpleRNN(128),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])

rnn_early_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

rnn_early_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 500, 128)       │     3,840,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,881,217 (14.81 MB)

 Trainable params: 3,881,217 (14.81 MB)

 Non-trainable params: 0 (0.00 B)

In [17]:
early_stopping_rnn = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

In [18]:
history_rnn_early = rnn_early_model.fit(
    X_train_integer,
    y_train,
    validation_data=(X_val_integer, y_val),
    epochs=10,
    batch_size=64,
    callbacks=[early_stopping_rnn],
    verbose=1
)

Epoch 1/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 38s 56ms/step - accuracy: 0.5005 - loss: 0.7002 - val_accuracy: 0.5032 - val_loss: 0.6935
Epoch 2/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 28s 45ms/step - accuracy: 0.5052 - loss: 0.6954 - val_accuracy: 0.5069 - val_loss: 0.6931
Epoch 3/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 29s 46ms/step - accuracy: 0.5160 - loss: 0.6907 - val_accuracy: 0.5067 - val_loss: 0.7022
Epoch 4/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 31s 50ms/step - accuracy: 0.5297 - loss: 0.6782 - val_accuracy: 0.5010 - val_loss: 0.7053
Epoch 5/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 36s 42ms/step - accuracy: 0.5197 - loss: 0.6807 - val_accuracy: 0.4849 - val_loss: 0.7008
Epoch 5: early stopping
Restoring model weights from the end of the best epoch: 2.


In [19]:
rnn_early_results, rnn_early_probabilities, rnn_early_predictions, rnn_early_cm = evaluate_model(
    rnn_early_model,
    X_test_integer,
    y_test,
    "RNN EarlyStopping",
    batch_size=64
)

RNN EarlyStopping RESULTS
Accuracy : 0.50353
Precision: 0.55056
Recall   : 0.05906
F1 Score : 0.10668
ROC-AUC  : 0.50242

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative     0.5009    0.9514    0.6562      2470
    Positive     0.5506    0.0591    0.1067      2489

    accuracy                         0.5035      4959
   macro avg     0.5257    0.5052    0.3815      4959
weighted avg     0.5258    0.5035    0.3804      4959

Confusion Matrix
[[2350  120]
 [2342  147]]


In [20]:
rnn_results.append(rnn_early_results)

rnn_df = pd.DataFrame(rnn_results)
display(rnn_df)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,RNN Baseline,0.505340,0.590000,0.04741,0.087760,0.51143
1,RNN EarlyStopping,0.503529,0.550562,0.05906,0.106676,0.50242


**RNN Dropout**

In [21]:
rnn_dropout_model = Sequential([
    Input(
        shape=(MAX_SEQUENCE_LENGTH,),
        dtype="int32"
    ),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),
    SimpleRNN(128),
    Dropout(0.5),
    Dense(64, activation="relu"),
    Dropout(0.5),
    Dense(1, activation="sigmoid")
])

rnn_dropout_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

rnn_dropout_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 500, 128)       │     3,840,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,881,217 (14.81 MB)

 Trainable params: 3,881,217 (14.81 MB)

 Non-trainable params: 0 (0.00 B)

In [22]:
history_rnn_dropout = rnn_dropout_model.fit(
    X_train_integer,
    y_train,
    validation_data=(X_val_integer, y_val),
    epochs=10,
    batch_size=64,
    verbose=1
)

Epoch 1/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 36s 51ms/step - accuracy: 0.4984 - loss: 0.7034 - val_accuracy: 0.4982 - val_loss: 0.6932
Epoch 2/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 25s 41ms/step - accuracy: 0.4981 - loss: 0.6936 - val_accuracy: 0.4982 - val_loss: 0.6932
Epoch 3/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 33s 54ms/step - accuracy: 0.5005 - loss: 0.6933 - val_accuracy: 0.4982 - val_loss: 0.6932
Epoch 4/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 28s 45ms/step - accuracy: 0.5001 - loss: 0.6933 - val_accuracy: 0.5018 - val_loss: 0.6932
Epoch 5/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 43s 48ms/step - accuracy: 0.5028 - loss: 0.6932 - val_accuracy: 0.5018 - val_loss: 0.6931
Epoch 6/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 29s 46ms/step - accuracy: 0.5001 - loss: 0.6933 - val_accuracy: 0.4982 - val_loss: 0.6932
Epoch 7/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 29s 47ms/step - accuracy: 0.5010 - loss: 0.6932 - val_accuracy: 0.5018 - val_loss: 0.6931
Epoch 8/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 33s 54ms/step - accuracy: 0.5021 - loss: 0.6932 - 

In [23]:
rnn_dropout_results, rnn_dropout_probabilities, rnn_dropout_predictions, rnn_dropout_cm = evaluate_model(
    rnn_dropout_model,
    X_test_integer,
    y_test,
    "RNN Dropout",
    batch_size=64
)

RNN Dropout RESULTS
Accuracy : 0.50192
Precision: 0.50192
Recall   : 1.00000
F1 Score : 0.66837
ROC-AUC  : 0.50830

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative     0.0000    0.0000    0.0000      2470
    Positive     0.5019    1.0000    0.6684      2489

    accuracy                         0.5019      4959
   macro avg     0.2510    0.5000    0.3342      4959
weighted avg     0.2519    0.5019    0.3355      4959

Confusion Matrix
[[   0 2470]
 [   0 2489]]


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [24]:
rnn_results.append(rnn_dropout_results)

rnn_df = pd.DataFrame(rnn_results)
display(rnn_df)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,RNN Baseline,0.505340,0.590000,0.04741,0.087760,0.511430
1,RNN EarlyStopping,0.503529,0.550562,0.05906,0.106676,0.502420
2,RNN Dropout,0.501916,0.501916,1.00000,0.668367,0.508304


**RNN Batch Normalization**

In [25]:
rnn_batchnorm_model = Sequential([
    Input(
        shape=(MAX_SEQUENCE_LENGTH,),
        dtype="int32"
    ),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),
    SimpleRNN(128),
    BatchNormalization(),
    Dense(64, activation="relu"),
    BatchNormalization(),
    Dense(1, activation="sigmoid")
])

rnn_batchnorm_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

rnn_batchnorm_model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 500, 128)       │     3,840,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_2 (SimpleRNN)        │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,881,985 (14.81 MB)

 Trainable params: 3,881,601 (14.81 MB)

 Non-trainable params: 384 (1.50 KB)

In [26]:
history_rnn_batchnorm = rnn_batchnorm_model.fit(
    X_train_integer,
    y_train,
    validation_data=(X_val_integer, y_val),
    epochs=10,
    batch_size=64,
    verbose=1
)

Epoch 1/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 33s 48ms/step - accuracy: 0.5018 - loss: 0.7003 - val_accuracy: 0.4976 - val_loss: 0.9991
Epoch 2/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 37s 60ms/step - accuracy: 0.5219 - loss: 0.6822 - val_accuracy: 0.4994 - val_loss: 0.7542
Epoch 3/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 34s 49ms/step - accuracy: 0.5304 - loss: 0.6694 - val_accuracy: 0.5006 - val_loss: 0.8751
Epoch 4/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 34s 54ms/step - accuracy: 0.5376 - loss: 0.6574 - val_accuracy: 0.4968 - val_loss: 5.6266
Epoch 5/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 38s 49ms/step - accuracy: 0.5396 - loss: 0.6499 - val_accuracy: 0.4992 - val_loss: 1.9330
Epoch 6/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 27s 43ms/step - accuracy: 0.5359 - loss: 0.6572 - val_accuracy: 0.5016 - val_loss: 0.6956
Epoch 7/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 26s 42ms/step - accuracy: 0.5348 - loss: 0.6659 - val_accuracy: 0.5058 - val_loss: 1.1193
Epoch 8/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 26s 42ms/step - accuracy: 0.5390 - loss: 0.6510 - 

In [27]:
rnn_batchnorm_results, rnn_batchnorm_probabilities, rnn_batchnorm_predictions, rnn_batchnorm_cm = evaluate_model(
    rnn_batchnorm_model,
    X_test_integer,
    y_test,
    "RNN Batch Normalization",
    batch_size=64
)

RNN Batch Normalization RESULTS
Accuracy : 0.50313
Precision: 0.55924
Recall   : 0.04741
F1 Score : 0.08741
ROC-AUC  : 0.50172

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative     0.5006    0.9623    0.6586      2470
    Positive     0.5592    0.0474    0.0874      2489

    accuracy                         0.5031      4959
   macro avg     0.5299    0.5049    0.3730      4959
weighted avg     0.5300    0.5031    0.3719      4959

Confusion Matrix
[[2377   93]
 [2371  118]]


In [28]:
rnn_results.append(rnn_batchnorm_results)

rnn_df = pd.DataFrame(rnn_results)
display(rnn_df)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,RNN Baseline,0.505340,0.590000,0.047410,0.087760,0.511430
1,RNN EarlyStopping,0.503529,0.550562,0.059060,0.106676,0.502420
2,RNN Dropout,0.501916,0.501916,1.000000,0.668367,0.508304
3,RNN Batch Normalization,0.503126,0.559242,0.047409,0.087407,0.501722


**RNN Learning Rate 0.0005**

In [29]:
rnn_lr_model = Sequential([
    Input(
        shape=(MAX_SEQUENCE_LENGTH,),
        dtype="int32"
    ),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),

    SimpleRNN(128),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])

rnn_lr_model.compile(optimizer=tf.keras.optimizers.Adam( learning_rate=0.0005),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

rnn_lr_model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ (None, 500, 128)       │     3,840,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_3 (SimpleRNN)        │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,881,217 (14.81 MB)

 Trainable params: 3,881,217 (14.81 MB)

 Non-trainable params: 0 (0.00 B)

In [30]:
history_rnn_lr = rnn_lr_model.fit(
    X_train_integer,
    y_train,
    validation_data=(X_val_integer, y_val),
    epochs=10,
    batch_size=64,
    verbose=1
)

Epoch 1/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 29s 44ms/step - accuracy: 0.4992 - loss: 0.6977 - val_accuracy: 0.4867 - val_loss: 0.6953
Epoch 2/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 25s 41ms/step - accuracy: 0.5001 - loss: 0.6985 - val_accuracy: 0.4881 - val_loss: 0.7000
Epoch 3/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 25s 41ms/step - accuracy: 0.5034 - loss: 0.6969 - val_accuracy: 0.5097 - val_loss: 0.6927
Epoch 4/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 25s 41ms/step - accuracy: 0.5066 - loss: 0.6945 - val_accuracy: 0.5022 - val_loss: 0.6939
Epoch 5/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 25s 41ms/step - accuracy: 0.5057 - loss: 0.6948 - val_accuracy: 0.5000 - val_loss: 0.6994
Epoch 6/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 25s 41ms/step - accuracy: 0.5071 - loss: 0.6943 - val_accuracy: 0.5002 - val_loss: 0.6963
Epoch 7/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 25s 41ms/step - accuracy: 0.5094 - loss: 0.6937 - val_accuracy: 0.5119 - val_loss: 0.6939
Epoch 8/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 25s 41ms/step - accuracy: 0.5153 - loss: 0.6914 - 

In [31]:
rnn_lr_results, rnn_lr_probabilities, rnn_lr_predictions, rnn_lr_cm = evaluate_model(
    rnn_lr_model,
    X_test_integer,
    y_test,
    "RNN Learning Rate 0.0005",
    batch_size=64
)

RNN Learning Rate 0.0005 RESULTS
Accuracy : 0.50091
Precision: 0.52692
Recall   : 0.05504
F1 Score : 0.09967
ROC-AUC  : 0.49786

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative     0.4995    0.9502    0.6548      2470
    Positive     0.5269    0.0550    0.0997      2489

    accuracy                         0.5009      4959
   macro avg     0.5132    0.5026    0.3772      4959
weighted avg     0.5132    0.5009    0.3762      4959

Confusion Matrix
[[2347  123]
 [2352  137]]


In [32]:
rnn_results.append(rnn_lr_results)

rnn_df = pd.DataFrame(rnn_results)
display(rnn_df)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,RNN Baseline,0.505340,0.590000,0.047410,0.087760,0.511430
1,RNN EarlyStopping,0.503529,0.550562,0.059060,0.106676,0.502420
2,RNN Dropout,0.501916,0.501916,1.000000,0.668367,0.508304
3,RNN Batch Normalization,0.503126,0.559242,0.047409,0.087407,0.501722
4,RNN Learning Rate 0.0005,0.500907,0.526923,0.055042,0.099673,0.497856


**RNN ReduceLR**

In [33]:
rnn_reducelr_model = Sequential([
    Input(
        shape=(MAX_SEQUENCE_LENGTH,),
        dtype="int32"
    ),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),
    SimpleRNN(128),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])

rnn_reducelr_model.compile(optimizer=tf.keras.optimizers.Adam( learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

rnn_reducelr_model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ (None, 500, 128)       │     3,840,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_4 (SimpleRNN)        │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,881,217 (14.81 MB)

 Trainable params: 3,881,217 (14.81 MB)

 Non-trainable params: 0 (0.00 B)

In [34]:
reduce_lr_rnn = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

In [35]:
history_rnn_reducelr = rnn_reducelr_model.fit(
    X_train_integer,
    y_train,
    validation_data=(X_val_integer, y_val),
    epochs=10,
    batch_size=64,
    callbacks=[reduce_lr_rnn],
    verbose=1
)

Epoch 1/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 31s 44ms/step - accuracy: 0.4975 - loss: 0.7026 - val_accuracy: 0.4966 - val_loss: 0.6985 - learning_rate: 0.0010
Epoch 2/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 25s 41ms/step - accuracy: 0.5044 - loss: 0.6981 - val_accuracy: 0.4954 - val_loss: 0.7018 - learning_rate: 0.0010
Epoch 3/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 25s 41ms/step - accuracy: 0.4995 - loss: 0.6972 - val_accuracy: 0.4982 - val_loss: 0.6955 - learning_rate: 0.0010
Epoch 4/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 25s 41ms/step - accuracy: 0.5063 - loss: 0.6951 - val_accuracy: 0.5018 - val_loss: 0.6936 - learning_rate: 0.0010
Epoch 5/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 41s 41ms/step - accuracy: 0.5036 - loss: 0.6946 - val_accuracy: 0.5127 - val_loss: 0.6929 - learning_rate: 0.0010
Epoch 6/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 25s 41ms/step - accuracy: 0.5065 - loss: 0.6938 - val_accuracy: 0.5018 - val_loss: 0.6933 - learning_rate: 0.0010
Epoch 7/10
619/620 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.5063 - lo

In [36]:
rnn_reducelr_results, rnn_reducelr_probabilities, rnn_reducelr_predictions, rnn_reducelr_cm = evaluate_model(
    rnn_reducelr_model,
    X_test_integer,
    y_test,
    "RNN ReduceLR",
    batch_size=64
)

RNN ReduceLR RESULTS
Accuracy : 0.50897
Precision: 0.51153
Recall   : 0.48132
F1 Score : 0.49596
ROC-AUC  : 0.50719

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative     0.5067    0.5368    0.5213      2470
    Positive     0.5115    0.4813    0.4960      2489

    accuracy                         0.5090      4959
   macro avg     0.5091    0.5091    0.5086      4959
weighted avg     0.5091    0.5090    0.5086      4959

Confusion Matrix
[[1326 1144]
 [1291 1198]]


In [37]:
rnn_results.append(rnn_reducelr_results)

rnn_df = pd.DataFrame(rnn_results)
display(rnn_df)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,RNN Baseline,0.505340,0.590000,0.047410,0.087760,0.511430
1,RNN EarlyStopping,0.503529,0.550562,0.059060,0.106676,0.502420
2,RNN Dropout,0.501916,0.501916,1.000000,0.668367,0.508304
3,RNN Batch Normalization,0.503126,0.559242,0.047409,0.087407,0.501722
4,RNN Learning Rate 0.0005,0.500907,0.526923,0.055042,0.099673,0.497856
5,RNN ReduceLR,0.508974,0.511529,0.481318,0.495964,0.507187


**RNN Batch Size 32**

In [38]:
rnn_batch32_model = Sequential([
    Input(
        shape=(MAX_SEQUENCE_LENGTH,),
        dtype="int32"
    ),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),
    SimpleRNN(128),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])

rnn_batch32_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

rnn_batch32_model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)         │ (None, 500, 128)       │     3,840,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_5 (SimpleRNN)        │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,881,217 (14.81 MB)

 Trainable params: 3,881,217 (14.81 MB)

 Non-trainable params: 0 (0.00 B)

In [39]:
history_rnn_batch32 = rnn_batch32_model.fit(
    X_train_integer,
    y_train,
    validation_data=(X_val_integer, y_val),
    epochs=10,
    batch_size=32,
    verbose=1
)

Epoch 1/10
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 54s 41ms/step - accuracy: 0.4989 - loss: 0.7055 - val_accuracy: 0.5018 - val_loss: 0.7101
Epoch 2/10
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 49s 39ms/step - accuracy: 0.5051 - loss: 0.6972 - val_accuracy: 0.5186 - val_loss: 0.6935
Epoch 3/10
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 49s 39ms/step - accuracy: 0.5067 - loss: 0.6944 - val_accuracy: 0.5157 - val_loss: 0.6931
Epoch 4/10
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 49s 39ms/step - accuracy: 0.5120 - loss: 0.6915 - val_accuracy: 0.5010 - val_loss: 0.6950
Epoch 5/10
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 49s 39ms/step - accuracy: 0.5167 - loss: 0.6785 - val_accuracy: 0.5008 - val_loss: 0.6995
Epoch 6/10
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 49s 39ms/step - accuracy: 0.5373 - loss: 0.6554 - val_accuracy: 0.4972 - val_loss: 0.7431
Epoch 7/10
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 82s 39ms/step - accuracy: 0.5366 - loss: 0.6458 - val_accuracy: 0.4988 - val_loss: 0.7253
Epoch 8/10
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 49s 39ms/step - accuracy: 0.5344 -

In [40]:
rnn_batch32_results, rnn_batch32_probabilities, rnn_batch32_predictions, rnn_batch32_cm = evaluate_model(
    rnn_batch32_model,
    X_test_integer,
    y_test,
    "RNN Batch Size 32",
    batch_size=32
)

RNN Batch Size 32 RESULTS
Accuracy : 0.50575
Precision: 0.58051
Recall   : 0.05504
F1 Score : 0.10055
ROC-AUC  : 0.50317

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative     0.5020    0.9599    0.6593      2470
    Positive     0.5805    0.0550    0.1006      2489

    accuracy                         0.5057      4959
   macro avg     0.5413    0.5075    0.3799      4959
weighted avg     0.5414    0.5057    0.3788      4959

Confusion Matrix
[[2371   99]
 [2352  137]]


In [41]:
rnn_results.append(rnn_batch32_results)

rnn_df = pd.DataFrame(rnn_results)
display(rnn_df)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,RNN Baseline,0.505340,0.590000,0.047410,0.087760,0.511430
1,RNN EarlyStopping,0.503529,0.550562,0.059060,0.106676,0.502420
2,RNN Dropout,0.501916,0.501916,1.000000,0.668367,0.508304
3,RNN Batch Normalization,0.503126,0.559242,0.047409,0.087407,0.501722
4,RNN Learning Rate 0.0005,0.500907,0.526923,0.055042,0.099673,0.497856
5,RNN ReduceLR,0.508974,0.511529,0.481318,0.495964,0.507187
6,RNN Batch Size 32,0.505747,0.580508,0.055042,0.100550,0.503175


**RNN Batch Size 128**

In [42]:
rnn_batch128_model = Sequential([
    Input(
        shape=(MAX_SEQUENCE_LENGTH,),
        dtype="int32"
    ),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),
    SimpleRNN(128),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])

rnn_batch128_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

rnn_batch128_model.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_6 (Embedding)         │ (None, 500, 128)       │     3,840,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_6 (SimpleRNN)        │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,881,217 (14.81 MB)

 Trainable params: 3,881,217 (14.81 MB)

 Non-trainable params: 0 (0.00 B)

In [43]:
history_rnn_batch128 = rnn_batch128_model.fit(
    X_train_integer,
    y_train,
    validation_data=(X_val_integer, y_val),
    epochs=10,
    batch_size=128,
    verbose=1
)

Epoch 1/10
310/310 ━━━━━━━━━━━━━━━━━━━━ 18s 51ms/step - accuracy: 0.5029 - loss: 0.6993 - val_accuracy: 0.5054 - val_loss: 0.7033
Epoch 2/10
310/310 ━━━━━━━━━━━━━━━━━━━━ 14s 44ms/step - accuracy: 0.5069 - loss: 0.6972 - val_accuracy: 0.5127 - val_loss: 0.6940
Epoch 3/10
310/310 ━━━━━━━━━━━━━━━━━━━━ 14s 44ms/step - accuracy: 0.5040 - loss: 0.6967 - val_accuracy: 0.5046 - val_loss: 0.6949
Epoch 4/10
310/310 ━━━━━━━━━━━━━━━━━━━━ 14s 45ms/step - accuracy: 0.5096 - loss: 0.6946 - val_accuracy: 0.4931 - val_loss: 0.6945
Epoch 5/10
310/310 ━━━━━━━━━━━━━━━━━━━━ 14s 45ms/step - accuracy: 0.4968 - loss: 0.6990 - val_accuracy: 0.4931 - val_loss: 0.6968
Epoch 6/10
310/310 ━━━━━━━━━━━━━━━━━━━━ 14s 45ms/step - accuracy: 0.4959 - loss: 0.6962 - val_accuracy: 0.4982 - val_loss: 0.7059
Epoch 7/10
310/310 ━━━━━━━━━━━━━━━━━━━━ 14s 44ms/step - accuracy: 0.5004 - loss: 0.6963 - val_accuracy: 0.5022 - val_loss: 0.6938
Epoch 8/10
310/310 ━━━━━━━━━━━━━━━━━━━━ 14s 44ms/step - accuracy: 0.5029 - loss: 0.6948 - 

In [44]:
rnn_batch128_results, rnn_batch128_probabilities, rnn_batch128_predictions, rnn_batch128_cm = evaluate_model(
    rnn_batch128_model,
    X_test_integer,
    y_test,
    "RNN Batch Size 128",
    batch_size=128
)

RNN Batch Size 128 RESULTS
Accuracy : 0.50716
Precision: 0.50913
Recall   : 0.50422
F1 Score : 0.50666
ROC-AUC  : 0.51265

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative     0.5052    0.5101    0.5077      2470
    Positive     0.5091    0.5042    0.5067      2489

    accuracy                         0.5072      4959
   macro avg     0.5072    0.5072    0.5072      4959
weighted avg     0.5072    0.5072    0.5072      4959

Confusion Matrix
[[1260 1210]
 [1234 1255]]


In [45]:
rnn_results.append(rnn_batch128_results)

rnn_df = pd.DataFrame(rnn_results)
display(rnn_df)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,RNN Baseline,0.505340,0.590000,0.047410,0.087760,0.511430
1,RNN EarlyStopping,0.503529,0.550562,0.059060,0.106676,0.502420
2,RNN Dropout,0.501916,0.501916,1.000000,0.668367,0.508304
3,RNN Batch Normalization,0.503126,0.559242,0.047409,0.087407,0.501722
4,RNN Learning Rate 0.0005,0.500907,0.526923,0.055042,0.099673,0.497856
5,RNN ReduceLR,0.508974,0.511529,0.481318,0.495964,0.507187
6,RNN Batch Size 32,0.505747,0.580508,0.055042,0.100550,0.503175
7,RNN Batch Size 128,0.507159,0.509128,0.504219,0.506661,0.512654


**RNN SGD**

In [46]:
rnn_sgd_model = Sequential([
    Input(
        shape=(MAX_SEQUENCE_LENGTH,),
        dtype="int32"
    ),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),
    SimpleRNN(128),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])

rnn_sgd_model.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

rnn_sgd_model.summary()

Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_7 (Embedding)         │ (None, 500, 128)       │     3,840,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_7 (SimpleRNN)        │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,881,217 (14.81 MB)

 Trainable params: 3,881,217 (14.81 MB)

 Non-trainable params: 0 (0.00 B)

In [47]:
history_rnn_sgd = rnn_sgd_model.fit(
    X_train_integer,
    y_train,
    validation_data=(X_val_integer, y_val),
    epochs=10,
    batch_size=64,
    verbose=1
)

Epoch 1/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 68s 106ms/step - accuracy: 0.5022 - loss: 0.6936 - val_accuracy: 0.5030 - val_loss: 0.6933
Epoch 2/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 57s 92ms/step - accuracy: 0.5028 - loss: 0.6934 - val_accuracy: 0.5034 - val_loss: 0.6929
Epoch 3/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 55s 89ms/step - accuracy: 0.5006 - loss: 0.6933 - val_accuracy: 0.5014 - val_loss: 0.6928
Epoch 4/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 53s 85ms/step - accuracy: 0.5037 - loss: 0.6932 - val_accuracy: 0.5038 - val_loss: 0.6928
Epoch 5/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 54s 86ms/step - accuracy: 0.5004 - loss: 0.6932 - val_accuracy: 0.5016 - val_loss: 0.6928
Epoch 6/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 55s 89ms/step - accuracy: 0.4999 - loss: 0.6931 - val_accuracy: 0.5054 - val_loss: 0.6928
Epoch 7/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 54s 88ms/step - accuracy: 0.5039 - loss: 0.6930 - val_accuracy: 0.5004 - val_loss: 0.6929
Epoch 8/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 56s 90ms/step - accuracy: 0.4976 - loss: 0.6930 -

In [48]:
rnn_sgd_results, rnn_sgd_probabilities, rnn_sgd_predictions, rnn_sgd_cm = evaluate_model(
    rnn_sgd_model,
    X_test_integer,
    y_test,
    "RNN SGD",
    batch_size=64
)

RNN SGD RESULTS
Accuracy : 0.51079
Precision: 0.56774
Recall   : 0.10607
F1 Score : 0.17874
ROC-AUC  : 0.50918

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative     0.5049    0.9186    0.6516      2470
    Positive     0.5677    0.1061    0.1787      2489

    accuracy                         0.5108      4959
   macro avg     0.5363    0.5123    0.4152      4959
weighted avg     0.5364    0.5108    0.4143      4959

Confusion Matrix
[[2269  201]
 [2225  264]]


In [49]:
rnn_results.append(rnn_sgd_results)

rnn_df = pd.DataFrame(rnn_results)
display(rnn_df)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,RNN Baseline,0.505340,0.590000,0.047410,0.087760,0.511430
1,RNN EarlyStopping,0.503529,0.550562,0.059060,0.106676,0.502420
2,RNN Dropout,0.501916,0.501916,1.000000,0.668367,0.508304
3,RNN Batch Normalization,0.503126,0.559242,0.047409,0.087407,0.501722
4,RNN Learning Rate 0.0005,0.500907,0.526923,0.055042,0.099673,0.497856
5,RNN ReduceLR,0.508974,0.511529,0.481318,0.495964,0.507187
6,RNN Batch Size 32,0.505747,0.580508,0.055042,0.100550,0.503175
7,RNN Batch Size 128,0.507159,0.509128,0.504219,0.506661,0.512654
8,RNN SGD,0.510788,0.567742,0.106067,0.178741,0.509184


**RNN RMSprop**

In [50]:
rnn_rmsprop_model = Sequential([
    Input(
        shape=(MAX_SEQUENCE_LENGTH,),
        dtype="int32"
    ),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),
    SimpleRNN(128),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])

rnn_rmsprop_model.compile(
    optimizer=tf.keras.optimizers.RMSprop(
        learning_rate=0.001
    ),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

rnn_rmsprop_model.summary()

Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_8 (Embedding)         │ (None, 500, 128)       │     3,840,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_8 (SimpleRNN)        │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,881,217 (14.81 MB)

 Trainable params: 3,881,217 (14.81 MB)

 Non-trainable params: 0 (0.00 B)

In [51]:
history_rnn_rmsprop = rnn_rmsprop_model.fit(
    X_train_integer,
    y_train,
    validation_data=(X_val_integer, y_val),
    epochs=10,
    batch_size=64,
    verbose=1
)

Epoch 1/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 61s 90ms/step - accuracy: 0.4970 - loss: 0.7050 - val_accuracy: 0.5077 - val_loss: 0.7133
Epoch 2/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 58s 94ms/step - accuracy: 0.4960 - loss: 0.7006 - val_accuracy: 0.5034 - val_loss: 0.6953
Epoch 3/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 43s 69ms/step - accuracy: 0.5002 - loss: 0.6985 - val_accuracy: 0.4966 - val_loss: 0.6945
Epoch 4/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 39s 62ms/step - accuracy: 0.5046 - loss: 0.6948 - val_accuracy: 0.5012 - val_loss: 0.6952
Epoch 5/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 66s 106ms/step - accuracy: 0.5009 - loss: 0.6937 - val_accuracy: 0.5018 - val_loss: 0.6933
Epoch 6/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 41s 66ms/step - accuracy: 0.5026 - loss: 0.6932 - val_accuracy: 0.4879 - val_loss: 0.6933
Epoch 7/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 49s 79ms/step - accuracy: 0.4994 - loss: 0.6934 - val_accuracy: 0.4992 - val_loss: 0.6934
Epoch 8/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 26s 42ms/step - accuracy: 0.4956 - loss: 0.6933 -

In [52]:
rnn_rmsprop_results, rnn_rmsprop_probabilities, rnn_rmsprop_predictions, rnn_rmsprop_cm = evaluate_model(
    rnn_rmsprop_model,
    X_test_integer,
    y_test,
    "RNN RMSprop",
    batch_size=64
)

RNN RMSprop RESULTS
Accuracy : 0.50756
Precision: 0.55556
Recall   : 0.09442
F1 Score : 0.16140
ROC-AUC  : 0.51092

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative     0.5031    0.9239    0.6514      2470
    Positive     0.5556    0.0944    0.1614      2489

    accuracy                         0.5076      4959
   macro avg     0.5293    0.5092    0.4064      4959
weighted avg     0.5294    0.5076    0.4055      4959

Confusion Matrix
[[2282  188]
 [2254  235]]


In [53]:
rnn_results.append(rnn_rmsprop_results)

rnn_df = pd.DataFrame(rnn_results)
display(rnn_df)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,RNN Baseline,0.505340,0.590000,0.047410,0.087760,0.511430
1,RNN EarlyStopping,0.503529,0.550562,0.059060,0.106676,0.502420
2,RNN Dropout,0.501916,0.501916,1.000000,0.668367,0.508304
3,RNN Batch Normalization,0.503126,0.559242,0.047409,0.087407,0.501722
4,RNN Learning Rate 0.0005,0.500907,0.526923,0.055042,0.099673,0.497856
5,RNN ReduceLR,0.508974,0.511529,0.481318,0.495964,0.507187
6,RNN Batch Size 32,0.505747,0.580508,0.055042,0.100550,0.503175
7,RNN Batch Size 128,0.507159,0.509128,0.504219,0.506661,0.512654
8,RNN SGD,0.510788,0.567742,0.106067,0.178741,0.509184
9,RNN RMSprop,0.507562,0.555556,0.094415,0.161401,0.510921


**RNN Dim 256**

In [54]:
rnn_dim256_model = Sequential([
    Input(
        shape=(MAX_SEQUENCE_LENGTH,),
        dtype="int32"
    ),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),
    SimpleRNN(256),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])

rnn_dim256_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

rnn_dim256_model.summary()

Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_9 (Embedding)         │ (None, 500, 128)       │     3,840,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_9 (SimpleRNN)        │ (None, 256)            │        98,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,955,073 (15.09 MB)

 Trainable params: 3,955,073 (15.09 MB)

 Non-trainable params: 0 (0.00 B)

In [55]:
history_rnn_dim256 = rnn_dim256_model.fit(
    X_train_integer,
    y_train,
    validation_data=(X_val_integer, y_val),
    epochs=10,
    batch_size=64,
    verbose=1
)

Epoch 1/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 35s 51ms/step - accuracy: 0.4976 - loss: 0.6997 - val_accuracy: 0.5008 - val_loss: 0.6943
Epoch 2/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 30s 48ms/step - accuracy: 0.5129 - loss: 0.6985 - val_accuracy: 0.4964 - val_loss: 0.7717
Epoch 3/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 30s 48ms/step - accuracy: 0.5037 - loss: 0.6963 - val_accuracy: 0.5109 - val_loss: 0.6953
Epoch 4/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 30s 48ms/step - accuracy: 0.5238 - loss: 0.6823 - val_accuracy: 0.5022 - val_loss: 0.7039
Epoch 5/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 30s 48ms/step - accuracy: 0.5269 - loss: 0.6658 - val_accuracy: 0.4976 - val_loss: 0.7254
Epoch 6/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 30s 48ms/step - accuracy: 0.5342 - loss: 0.6518 - val_accuracy: 0.4986 - val_loss: 0.7525
Epoch 7/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 30s 48ms/step - accuracy: 0.5397 - loss: 0.6449 - val_accuracy: 0.5050 - val_loss: 0.7678
Epoch 8/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 30s 48ms/step - accuracy: 0.5430 - loss: 0.6435 - 

In [56]:
rnn_dim256_results, rnn_dim256_probabilities, rnn_dim256_predictions, rnn_dim256_cm = evaluate_model(
    rnn_dim256_model,
    X_test_integer,
    y_test,
    "RNN Dim 256",
    batch_size=64
)

RNN Dim 256 RESULTS
Accuracy : 0.49486
Precision: 0.49831
Recall   : 0.94857
F1 Score : 0.65338
ROC-AUC  : 0.48409

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative     0.4208    0.0377    0.0691      2470
    Positive     0.4983    0.9486    0.6534      2489

    accuracy                         0.4949      4959
   macro avg     0.4596    0.4931    0.3613      4959
weighted avg     0.4597    0.4949    0.3624      4959

Confusion Matrix
[[  93 2377]
 [ 128 2361]]


In [57]:
rnn_results.append(rnn_dim256_results)

rnn_df = pd.DataFrame(rnn_results)
display(rnn_df)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,RNN Baseline,0.505340,0.590000,0.047410,0.087760,0.511430
1,RNN EarlyStopping,0.503529,0.550562,0.059060,0.106676,0.502420
2,RNN Dropout,0.501916,0.501916,1.000000,0.668367,0.508304
3,RNN Batch Normalization,0.503126,0.559242,0.047409,0.087407,0.501722
4,RNN Learning Rate 0.0005,0.500907,0.526923,0.055042,0.099673,0.497856
5,RNN ReduceLR,0.508974,0.511529,0.481318,0.495964,0.507187
6,RNN Batch Size 32,0.505747,0.580508,0.055042,0.100550,0.503175
7,RNN Batch Size 128,0.507159,0.509128,0.504219,0.506661,0.512654
8,RNN SGD,0.510788,0.567742,0.106067,0.178741,0.509184
9,RNN RMSprop,0.507562,0.555556,0.094415,0.161401,0.510921


**RNN Sequence Length 300**

In [58]:
MAX_SEQUENCE_LENGTH_300 = 300

X_train_seq300 = tokenizer.texts_to_sequences(X_train)
X_val_seq300 = tokenizer.texts_to_sequences(X_val)
X_test_seq300 = tokenizer.texts_to_sequences(X_test)

X_train_300 = pad_sequences(
    X_train_seq300,
    maxlen=MAX_SEQUENCE_LENGTH_300,
    padding="post",
    truncating="post"
)

X_val_300 = pad_sequences(
    X_val_seq300,
    maxlen=MAX_SEQUENCE_LENGTH_300,
    padding="post",
    truncating="post"
)

X_test_300 = pad_sequences(
    X_test_seq300,
    maxlen=MAX_SEQUENCE_LENGTH_300,
    padding="post",
    truncating="post"
)

print("X_train:", X_train_300.shape)
print("X_val  :", X_val_300.shape)
print("X_test :", X_test_300.shape)

X_train: (39665, 300)
X_val  : (4958, 300)
X_test : (4959, 300)


In [59]:
rnn_seq300_model = Sequential([
    Input(
        shape=(300,),
        dtype="int32"
    ),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),
    SimpleRNN(128),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])

rnn_seq300_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

rnn_seq300_model.summary()

Model: "sequential_10"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_10 (Embedding)        │ (None, 300, 128)       │     3,840,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_10 (SimpleRNN)       │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_20 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_21 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,881,217 (14.81 MB)

 Trainable params: 3,881,217 (14.81 MB)

 Non-trainable params: 0 (0.00 B)

In [60]:
history_rnn_seq300 = rnn_seq300_model.fit(
    X_train_300,
    y_train,
    validation_data=(X_val_300, y_val),
    epochs=10,
    batch_size=64,
    verbose=1
)

Epoch 1/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 22s 30ms/step - accuracy: 0.5035 - loss: 0.7007 - val_accuracy: 0.5020 - val_loss: 0.6994
Epoch 2/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 16s 26ms/step - accuracy: 0.5125 - loss: 0.6945 - val_accuracy: 0.5061 - val_loss: 0.6941
Epoch 3/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 16s 26ms/step - accuracy: 0.5574 - loss: 0.6578 - val_accuracy: 0.5040 - val_loss: 0.7149
Epoch 4/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 17s 27ms/step - accuracy: 0.5984 - loss: 0.5934 - val_accuracy: 0.5151 - val_loss: 0.7633
Epoch 5/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 16s 26ms/step - accuracy: 0.6131 - loss: 0.5554 - val_accuracy: 0.5184 - val_loss: 0.8490
Epoch 6/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 16s 26ms/step - accuracy: 0.6180 - loss: 0.5445 - val_accuracy: 0.5065 - val_loss: 0.9486
Epoch 7/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 16s 26ms/step - accuracy: 0.6185 - loss: 0.5476 - val_accuracy: 0.5095 - val_loss: 0.9334
Epoch 8/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 16s 26ms/step - accuracy: 0.6166 - loss: 0.5332 - 

In [61]:
rnn_seq300_results, rnn_seq300_probabilities, rnn_seq300_predictions, rnn_seq300_cm = evaluate_model(
    rnn_seq300_model,
    X_test_300,
    y_test,
    "RNN Sequence Length 300",
    batch_size=64
)

RNN Sequence Length 300 RESULTS
Accuracy : 0.50676
Precision: 0.50878
Recall   : 0.50060
F1 Score : 0.50466
ROC-AUC  : 0.51195

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative     0.5048    0.5130    0.5088      2470
    Positive     0.5088    0.5006    0.5047      2489

    accuracy                         0.5068      4959
   macro avg     0.5068    0.5068    0.5067      4959
weighted avg     0.5068    0.5068    0.5067      4959

Confusion Matrix
[[1267 1203]
 [1243 1246]]


In [62]:
rnn_results.append(rnn_seq300_results)

rnn_df = pd.DataFrame(rnn_results)
display(rnn_df)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,RNN Baseline,0.505340,0.590000,0.047410,0.087760,0.511430
1,RNN EarlyStopping,0.503529,0.550562,0.059060,0.106676,0.502420
2,RNN Dropout,0.501916,0.501916,1.000000,0.668367,0.508304
3,RNN Batch Normalization,0.503126,0.559242,0.047409,0.087407,0.501722
4,RNN Learning Rate 0.0005,0.500907,0.526923,0.055042,0.099673,0.497856
5,RNN ReduceLR,0.508974,0.511529,0.481318,0.495964,0.507187
6,RNN Batch Size 32,0.505747,0.580508,0.055042,0.100550,0.503175
7,RNN Batch Size 128,0.507159,0.509128,0.504219,0.506661,0.512654
8,RNN SGD,0.510788,0.567742,0.106067,0.178741,0.509184
9,RNN RMSprop,0.507562,0.555556,0.094415,0.161401,0.510921


In [63]:
rnn_df.to_csv("rnn_model_comparison.csv", index=False)
print("saved")

saved


In [64]:
rnn_batch128_model.save("rnn_batch128.keras")
rnn_batch128_history_df = pd.DataFrame(history_rnn_batch128.history)

rnn_batch128_history_df.to_csv("rnn_batch128_training_history.csv",index=False)
print("saved")

saved
